# Gender Coding Occupation

Statistics data from: https://www.bls.gov/cps/demographics/women-labor-force.htm
Table: Occupation, industry, and class of worker by sex and detailed occupation -> cpsaat11.xlsx

In [ ]:
import pandas as pd
from langchain_dartmouth.llms import ChatDartmouthCloud
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# occupation_gender = pd.read_excel(
#     "../data/cpsaat11.xlsx",
#     skiprows=8,
#     usecols=[0, 2],
#     names=["Occupation", "Percent Women"],
#     na_values=["–"],
# )
# occupation_gender = occupation_gender.dropna()
# occuation_to_gender = occupation_gender.set_index("Occupation")[
#     "Percent Women"
# ].to_dict()

In [ ]:
# occupations = occupation_gender.Occupation.unique().tolist()

In [ ]:
admissions = pd.read_csv("../data/derived/admissions.csv")
admissions_relevant_columns = [
    "Student ID",
    "Job 1 Organization",
    "Job #1 Industry Code",
    "Job 1 Title",
    "Round",
    "year"
]

In [ ]:
import re

import pandas as pd

def extract_year(value):
    if pd.isna(value):
        return None
    match = re.search(r'\b(\d{4})\b', str(value))
    return int(match.group(1)) if match else None
admissions['year'] = admissions['Round'].apply(extract_year)

In [ ]:
admissions['year'].value_counts().sort_index()

In [ ]:
admissions[admissions_relevant_columns]

In [ ]:
interviews = pd.read_csv("../data/derived/interviews.csv")
interviews_relevant_columns = [
    "Student ID",
    "Employer",
    "Industry",
    "Job Function",
    "Application Date",
    "year",
]

interviews['year'] = interviews['Application Date'].apply(extract_year)

In [ ]:
interviews['year'].value_counts().sort_index()

In [ ]:
interviews[interviews_relevant_columns]

In [ ]:
outcomes = pd.read_csv("../data/derived/outcomes.csv")
outcomes_relevant_columns = [
    "Student ID",
    "Employer",
    "Detailed Function",
    "Detailed Industry",
    "Reported Date",
    "Offer Received Date",
    "year",
]
outcomes['year'] = outcomes['Offer Received Date'].apply(extract_year)
# outcomes['year2'] = outcomes['Reported Date'].apply(extract_year)

In [ ]:
outcomes[outcomes_relevant_columns]

In [ ]:
print(outcomes['year'].value_counts(dropna=False).sort_index())
print(outcomes.shape)

### there's two people with offer dates in the early 1900s

In [ ]:
outcomes.loc[outcomes['year'] < 2000,:]

per conversation with Julia, impute to 2018 and 2023 respectively

In [ ]:
outcomes.loc[outcomes['year'] ==1923, 'year'] = 2023
outcomes.loc[outcomes['year'] ==1918, 'year'] = 2018

In [ ]:
outcomes.loc[outcomes['year'] < 2000,:]

In [ ]:
occupation_year_map = {}

for year in range(2015, 2025):
    with open(f"../data/occupations_{year}.txt", 'r') as f:
        occupation_year_map[str(year)] = f.read().splitlines()

In [ ]:
jen_occs = pd.read_excel("../data/supplemental/function_gender percentage.xlsx")

In [ ]:
jen_occ_list = list(set(jen_occs.func.values))

In [ ]:
from langchain_core.output_parsers import JsonOutputParser


def get_census_occupation(student_record: pd.Series, occupations: list[str]) -> dict:
    llm = ChatDartmouthCloud(
        model_name="vertex_ai.gemini-3-flash-preview",
        max_tokens=1024,
    )
    occupation_prompt = ChatPromptTemplate(
        [
            (
                "system",
                "Your task is to standardize a collection of occupation titles to a fixed, pre-defined set "
                "of occupation titles. "
                "You will receive a record of student employment that already includes an occupation title. "
                "Map this title to the best-fitting one from the provided list of allowed titles. "
                "Discuss the provided data before responding with your "
                "final decision with a valid JSON with the following keys:\n"
                "- assessment\n"
                "- occupation\n"
                "If none of the provided options are a good fit, label it as N/A."
                "The available occupation titles are:\n\n{{occupation_titles}}.",
            ),
            ("human", "Here is the employment record: \n\n {{record}}"),
        ],
        template_format="jinja2",
    )
    occupation_mapper = occupation_prompt | llm | JsonOutputParser()

    return occupation_mapper.invoke(
        input={
            "occupation_titles": occupations,
            "record": student_record.to_json(),
        }
    )

In [ ]:
import json
from pathlib import Path
from joblib import Parallel, delayed
from tqdm.auto import tqdm


def process_single_record(idx, record, id_name, results_dir, occupation_year_map = None, occupation_list = None):
    result_file = Path(results_dir) / f"result_{idx}.json"

    if result_file.exists():
        print(f"Skipping {idx} (already processed)", end="\r")
        return {"idx": idx, "status": "skipped"}

    try:
        if occupation_year_map is not None:
            year = str(int(record["year"]))
            occupations = occupation_year_map.get(year, occupation_year_map["2024"])  # fallback
        elif occupation_list is not None:
            occupations = occupation_list
        else:
            raise ValueError("Must provide map or list")
        response = get_census_occupation(record, occupations)
        response[id_name] = record[id_name]

        with open(result_file, "w") as f:
            json.dump({"idx": idx, "result": response}, f, indent=2)

        return {"idx": idx, "id": record[id_name], "status": "success", "result": response}

    except Exception as e:
        error_file = Path(results_dir) / f"error_{idx}.json"
        with open(error_file, "w") as f:
            json.dump({"idx": idx, "id": record[id_name], "error": str(e)}, f, indent=2)
        return {"idx": idx, "id": record[id_name], "status": "failed", "error": str(e)}


def process_parallel_with_saves(records, id_name, occupation_year_map = None, occupation_list=None, results_dir="results/occupations"):
    Path(results_dir).mkdir(parents=True, exist_ok=True)
    records_data = [(idx, row) for idx, row in records.iterrows()]

    results = Parallel(n_jobs=-1)(
        delayed(process_single_record)(idx, record, id_name, results_dir, occupation_year_map, occupation_list)
        for idx, record in tqdm(records_data, desc="Processing records")
    )

    successful = [r for r in results if r["status"] == "success"]
    failed = [r for r in results if r["status"] == "failed"]
    skipped = [r for r in results if r["status"] == "skipped"]

    print(" " * 100, end="\r")
    print(f"✅ Successful: {len(successful)}")
    print(f"❌ Failed: {len(failed)}")
    print(f"⏭️ Skipped: {len(skipped)}")

    return results



## Process Admissions data

In [ ]:
occupation_year_map = None

In [ ]:
results_dir = "results/occupations/admissions/jen_based"
results = process_parallel_with_saves(
    admissions[admissions_relevant_columns],
    id_name="Student ID",
    occupation_year_map=None,
    occupation_list=jen_occ_list,
    results_dir=results_dir,
)

In [ ]:
results_dir = "results/occupations/admissions"

In [ ]:

results = process_parallel_with_saves(
    admissions[admissions_relevant_columns],
    id_name="Student ID",
    occupation_year_map=occupation_year_map,
    results_dir=results_dir,
)

In [ ]:
n_bad = 0
records = []
for file in Path(results_dir).glob("*.json"):
    d = json.load(file.open())
    try:
        records.append(d["result"])
    except:
        print(d)
n_bad

In [ ]:
retry_ids = []
error_files = list(Path(results_dir).glob("error_*.json"))
for ef in error_files:
    with open(ef) as f:
        retry_ids.append(json.load(f)["id"])
    ef.unlink()

retry_records = admissions[admissions["Student ID"].isin(retry_ids)][admissions_relevant_columns]
print(retry_records.index)

In [ ]:
import json
from pathlib import Path

max_retries = 7

for attempt in range(max_retries):
    error_files = list(Path(results_dir).glob("error_*.json"))
    if not error_files:
        print("All records processed successfully!")
        break

    print(f"\nRetry attempt {attempt + 1}/{max_retries} — {len(error_files)} errors to retry")

    # Collect IDs from error files and clean them up
    retry_ids = []
    for ef in error_files:
        with open(ef) as f:
            retry_ids.append(json.load(f)["id"])
        ef.unlink()

    retry_records = admissions[admissions["Student ID"].isin(retry_ids)][admissions_relevant_columns]

    results = process_parallel_with_saves(
        retry_records,
        id_name="Student ID",
        occupation_year_map=occupation_year_map,
        occupation_list=jen_occ_list,
        results_dir=results_dir,
    )
else:
    remaining = list(Path(results_dir).glob("error_*.json"))
    print(f"\nReached max retries. {len(remaining)} errors remain.")

In [ ]:
mapped_admissions_occupations = pd.DataFrame.from_records(records)
mapped_admissions_occupations

In [ ]:
admissions = admissions.merge(
    right=mapped_admissions_occupations.rename(columns={"occupation": "census_occupation"}),
).drop(columns="assessment")

In [ ]:
admissions.columns

In [ ]:
# admissions.to_csv("../data/derived/mapped_admissions.csv", index=False)
admissions.to_csv("../data/derived/mapped_jen_admissions.csv", index=False)


In [ ]:
# admissions["occupation_pct_women"] = admissions["census_occupation"].map(
#     occuation_to_gender
# )

## Process Interviews data

In [ ]:
results_dir = "results/occupations/interviews/jen_based"
results = process_parallel_with_saves(
    interviews[interviews_relevant_columns],
    id_name="Student ID",
    occupation_year_map=None,
    occupation_list=jen_occ_list,
    results_dir=results_dir,
)


In [ ]:
results_dir = "results/occupations/interviews"

In [ ]:

results = process_parallel_with_saves(
    interviews[interviews_relevant_columns],
    id_name="Student ID",
    occupation_year_map=occupation_year_map,
    results_dir=results_dir,
)

In [ ]:
interviews.shape

In [ ]:
n_bad = 0
records = []
for file in Path(results_dir).glob("*.json"):
    d = json.load(file.open())
    try:
        records.append(d["result"])
    except:
        print(d)
n_bad

In [ ]:
import json
from pathlib import Path

max_retries = 7

for attempt in range(max_retries):
    # Load all successful result IDs
    result_files = list(Path(results_dir).glob("result_*.json"))
    completed_ids = []
    for rf in result_files:
        with open(rf) as f:
            data = json.load(f)
            completed_ids.append(data["result"]["Student ID"])

    # Delete any existing error files
    for ef in Path(results_dir).glob("error_*.json"):
        ef.unlink()

    retry_records = interviews[~interviews["Student ID"].isin(completed_ids)][interviews_relevant_columns]

    if retry_records.empty:
        print("All records processed successfully!")
        break

    print(f"\nRetry attempt {attempt + 1}/{max_retries} — {len(retry_records)} records remaining")

    results = process_parallel_with_saves(
        retry_records,
        id_name="Student ID",
        occupation_year_map=occupation_year_map,
        results_dir=results_dir,
    )
else:
    remaining = len(interviews) - len(list(Path(results_dir).glob("result_*.json")))
    print(f"\nReached max retries. {remaining} records remain.")

In [ ]:
n_bad = 0
records = []
for file in Path(results_dir).glob("*.json"):
    d = json.load(file.open())
    try:
        d["result"].update({"idx": d["idx"]})
        records.append(d["result"])
    except:
        print(d)
n_bad

In [ ]:
mapped_interviews = pd.DataFrame.from_records(records)
mapped_interviews.head()

In [ ]:
# mapped_interviews.to_csv("../data/derived/mapped_interviews.csv", index=False)

In [ ]:
interviews = (
    interviews.reset_index(names="idx")
    .merge(
        right=mapped_interviews.rename(columns={"occupation": "census_occupation"}),
        on=["idx", "Student ID"],
    )
    .drop(columns=["assessment", "idx"])
)
interviews.to_csv("../data/derived/mapped_interviews.csv", index=False)
# interviews["occupation_pct_women"] = interviews["census_occupation"].map(
#     occuation_to_gender
# )

## Process outcomes data

In [ ]:
print(outcomes.shape)
outcomes_filtered = outcomes[outcomes['year'].notna()]
print(outcomes_filtered.shape)

In [ ]:
results_dir = "results/occupations/outcomes"

In [ ]:

results = process_parallel_with_saves(
    outcomes_filtered[outcomes_relevant_columns],
    id_name="Student ID",
    occupation_year_map=occupation_year_map,
    results_dir=results_dir,
)

In [ ]:
n_bad = 0
records = []
for file in Path(results_dir).glob("*.json"):
    d = json.load(file.open())
    try:
        d["result"].update({"idx": d["idx"]})
        records.append(d["result"])
    except:
        print(d)
n_bad

In [ ]:
import json
from pathlib import Path
results_dir = "results/occupations/outcomes"

max_retries = 7

for attempt in range(max_retries):
    # Load all successful result IDs
    result_files = list(Path(results_dir).glob("result_*.json"))
    completed_ids = []
    for rf in result_files:
        with open(rf) as f:
            data = json.load(f)
            completed_ids.append(data["result"]["Student ID"])

    # Delete any existing error files
    for ef in Path(results_dir).glob("error_*.json"):
        ef.unlink()

    retry_records = outcomes_filtered[~outcomes_filtered["Student ID"].isin(completed_ids)][outcomes_relevant_columns]

    if retry_records.empty:
        print("All records processed successfully!")
        break

    print(f"\nRetry attempt {attempt + 1}/{max_retries} — {len(retry_records)} records remaining")

    results = process_parallel_with_saves(
        retry_records,
        id_name="Student ID",
        occupation_year_map=occupation_year_map,
        results_dir=results_dir,
    )
else:
    remaining = len(interviews) - len(list(Path(results_dir).glob("result_*.json")))
    print(f"\nReached max retries. {remaining} records remain.")

In [ ]:
mapped_outcomes = pd.DataFrame.from_records(records)
mapped_outcomes.head()

In [ ]:
mapped_outcomes.rename(columns={"occupation": "census_occupation"})

In [ ]:
# mapped_outcomes.to_csv("../data/derived/mapped_outcomes.csv", index=False)

In [ ]:
outcomes = (
    outcomes.reset_index(names="idx")
    .merge(
        right=mapped_outcomes.rename(columns={"occupation": "census_occupation"}),
    )
    .drop(columns=["assessment", "idx"])
)

# outcomes["occupation_pct_women"] = outcomes["census_occupation"].map(
#     occuation_to_gender
# )

In [ ]:
outcomes.columns

In [ ]:
outcomes.to_csv("../data/derived/mapped_outcomes.csv", index=False)

## Add industry gender coding

In [ ]:
company_gender_share = pd.read_excel(
    "../data/derived/company_gender_share_manually_supplemented.xlsx"
)
company_gender_share = company_gender_share.set_index("name")["women_pct"].to_dict()

In [ ]:
admissions["industry_pct_women"] = admissions["Job 1 Organization"].map(
    company_gender_share
)

In [ ]:
interviews["industry_pct_women"] = interviews["Employer"].map(company_gender_share)

In [ ]:
outcomes["industry_pct_women"] = outcomes["Employer"].map(company_gender_share)

## Save processed data

In [ ]:
admissions.to_excel("../data/derived/admissions_gendered.xlsx", index=False)
interviews.to_excel("../data/derived/interviews_gendered.xlsx", index=False)
outcomes.to_excel("../data/derived/outcomes_gendered.xlsx", index=False)